# AI Resume Screening System with Tracing
**Innomatics Research Labs - Data Science Internship (Feb 2026)**  
Task 3: GenAI Assignment

In [62]:
# install required libraries
#!pip install langchain langchain-core langsmith huggingface_hub transformers langchain-community

In [63]:
import os
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

## Step 1: Setup API Keys and LangSmith Tracing

In [64]:
# paste your actual keys here
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN", "")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "resume-screening-task3"

print("Keys set.")
print("LangSmith project:", os.environ["LANGCHAIN_PROJECT"])

Keys set.
LangSmith project: resume-screening-task3


## Step 2: Job Description

In [65]:
job_description = """
Job Title: Data Scientist

Required Skills:
- Python programming
- Machine Learning (scikit-learn, XGBoost)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data manipulation
- Data visualization (Matplotlib, Seaborn, or Tableau)
- Statistics and probability
- NLP experience is a plus
- Cloud platforms (AWS, GCP, or Azure)

Experience Required: 2+ years in a data science or analytics role
Tools: Python, SQL, Jupyter Notebook, Git, scikit-learn, TensorFlow
"""

print("Job description loaded.")

Job description loaded.


## Step 3: Define the Three Resumes

In [66]:
# Strong candidate
resume_strong = """
Name: Aisha Khan
Experience: 3 years as Data Scientist at a fintech startup

Skills:
- Python (pandas, numpy, scikit-learn, XGBoost)
- Deep Learning with TensorFlow and Keras
- SQL - advanced queries, joins, window functions
- NLP - text classification and sentiment analysis
- Data visualization with Tableau and Seaborn
- Deployed models on AWS SageMaker
- Git and version control

Projects:
- Customer churn prediction model (XGBoost) - 92% accuracy
- NLP pipeline for ticket classification
- Tableau dashboards for business reporting
"""

# Average candidate
resume_average = """
Name: Ravi Sharma
Experience: 1.5 years as Junior Data Analyst

Skills:
- Python (pandas, matplotlib)
- Basic machine learning with scikit-learn
- SQL - basic queries and joins
- Excel and Google Sheets
- Jupyter Notebook

Projects:
- EDA on sales data
- Simple linear regression model for price prediction
- Basic visualizations for monthly reports
"""

# Weak candidate
resume_weak = """
Name: Priya Mehta
Experience: Fresher - recently completed BCA degree

Skills:
- Basic Python (loops, functions)
- HTML and CSS
- Microsoft Office (Word, Excel)
- Data entry

Projects:
- Library management system in Java
- Simple website using HTML and CSS
"""

print("All 3 resumes loaded.")

All 3 resumes loaded.


## Step 4: Create Prompts

In [67]:
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""You are a resume parser. Only extract what is written. Do NOT add anything.

Resume:
{resume}

List:
1. Skills
2. Years of experience
3. Tools used

Answer:"""
)

matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""Compare the candidate profile with the job description.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

List:
- Matching skills
- Missing skills
- Experience match (yes/no with reason)

Answer:"""
)

scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""Based on the match below, give a score from 0 to 100.

Match Analysis:
{match_result}

Job Description:
{job_description}

Rules:
- 80 to 100 if most skills match and experience is enough
- 50 to 79 if partial match
- 0 to 49 if poor match

Reply in this exact format:
Score: <number>
Reason: <one sentence>

Answer:"""
)

explanation_prompt = PromptTemplate(
    input_variables=["score_result", "match_result"],
    template="""Write a short recruiter-style summary for this candidate.

Score and Reason:
{score_result}

Match Analysis:
{match_result}

Write 2-3 sentences. Be honest. Do not assume anything not mentioned.

Answer:"""
)

print("All prompts created.")

All prompts created.


## Step 5: Setup LLM and Build Chains

Using a Hugging Face local model (`gpt2`) through Transformers + LangChain pipeline.

In [68]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id)
pipeline_obj = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
    truncation=True,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline_obj)
parser = StrOutputParser()

extraction_chain = extraction_prompt | llm | parser
matching_chain = matching_prompt | llm | parser
scoring_chain = scoring_prompt | llm | parser
explanation_chain = explanation_prompt | llm | parser

print("Hugging Face chains ready.")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11419.58it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Hugging Face chains ready.


## Step 6: Build the Screening Pipeline

In [69]:
def screen_resume(resume_text, jd_text, candidate_label="Candidate"):
    print(f"\n--- Running pipeline for: {candidate_label} ---")

    # step 1: extract skills from resume
    extracted = extraction_chain.invoke({"resume": resume_text})
    print("\nExtracted Info:")
    print(extracted)

    # step 2: match with job description
    match_result = matching_chain.invoke({
        "extracted_info": extracted,
        "job_description": jd_text
    })
    print("\nMatch Result:")
    print(match_result)

    # step 3: score the candidate
    score_result = scoring_chain.invoke({
        "match_result": match_result,
        "job_description": jd_text
    })
    print("\nScore:")
    print(score_result)

    # step 4: explain the result
    explanation = explanation_chain.invoke({
        "score_result": score_result,
        "match_result": match_result
    })
    print("\nFinal Explanation:")
    print(explanation)

    print("\n" + "="*60)

    return {
        "candidate": candidate_label,
        "extracted": extracted,
        "match": match_result,
        "score": score_result,
        "explanation": explanation
    }

print("Pipeline function defined.")

Pipeline function defined.


## Step 7: Run All 3 Resumes

Each run is automatically traced in LangSmith.

In [70]:
# Run 1 - Strong candidate
result_strong = screen_resume(resume_strong, job_description, candidate_label="Strong Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Strong Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


I know I have the skills for doing this job. I am not sure if it would be more practical for me to do this job. I would be interested in working with companies to get people to do this job, so please let me know what you think.

I have been doing this job for about three years now. I am a data scientist and I have worked on a lot of different projects. I like data science so I am very interested in what's coming up.

I am currently working on a project with a company called TensorFlow with an NLP


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:
 Yes or no


I'm currently in my mid-20s. I've been doing this job for about three years now. I am a data scientist and I have worked on a lot of different projects. I like data science so I am very interested in what's coming up.I am currently working on a project with a company called TensorFlow with an NLPJob Title: Data ScientistRequired Skills:- Python programming- Machine Learning (scikit-learn, XGBoost)- Deep Learning (TensorFlow or PyTorch)- SQL and data manipulation- NLP experience is


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:
 (as in "This is going to be interesting")

Reply: (as in "This is going to be interesting")

Reply: (as in "I've got my first job in a data science or analytics role")

Reply: (as in "This is going to be interesting")


Example of "matching up" for all jobs:


Hello,

The following job is going to be interesting.

I'm a data scientist in an AWS role. I have a lot of experience working on big projects and I've been working very hard

Final Explanation:


I am a data scientist. I am doing a lot of research into the data science fields. I want to make sure that people know my work and I am a data scientist. I have a lot of experience working on big projects and I've been working very hard.Match Analysis: Yes or noI'm currently in my mid-20s. I've been doing this job for about three years now. I am a data scientist in an AWS role. I have a lot of experience working on big projects and I've been working very hard.I am currently working on a project with



In [71]:
# Run 2 - Average candidate
result_average = screen_resume(resume_average, job_description, candidate_label="Average Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Average Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


You should get a good understanding of how data structures work. You should know how to write software that learns from experience and adapts to change. You will only learn the basics of how data structures work so that you can develop a data model that works.

You will learn how to write software that learns from experience and adapts to change. You will only learn the basics of how data structures work so that you can develop a data model that works. You will learn how to write software that learns from experience, and learn how to make data modeling more efficient. You will learn how


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:


- I had an internship at IBM. It was an internship. I taught SQL, Python, and TensorFlow, then I worked at IBM for over a year. I got my first job at IBM in 2008.

- I had an internship at IBM. It was an internship. I taught SQL, Python, and TensorFlow, then I worked at IBM for over a year. I got my first job at IBM in 2008. I have a 4 year career as a data scientist at IBM. I'm currently in the business of data visualization and visualization.

For


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:
 <1-7 sentences>

Comments: <no answer>

Comments for this answer: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99

Final Explanation:
 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99

Samples:

-

-

-

-

-




In [72]:
# Run 3 - Weak candidate
result_weak = screen_resume(resume_weak, job_description, candidate_label="Weak Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Weak Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


- Basic Python (loops, functions)

- HTML and CSS

- Microsoft Office (Word, Excel)

- Data entry

Projects:

- Library management system in Java

- Simple website using HTML and CSS

- Microsoft Office (Word, Excel)

- Data entry

Projects:

- Library management system in Java

- Simple website using HTML and CSS

- Microsoft Office (Word, Excel)

- Data entry

Projects:

- Library management system in Java



Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:
 Data science or analytics is a requirement. For more information about that, call

the Data Scientist Job Center

http://jobs.datschemas.org/

Job:

Job Description:

Job Title: Data Scientist

Required Skills:

- Python programming

- Machine Learning (scikit-learn, XGBoost)

- Deep Learning (TensorFlow or PyTorch)

- SQL and data manipulation

- Data visualization (Matplotlib, Seaborn, or Tableau)

- Statistics


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:
 <one sentence>

If you are looking for a position in a data science or analytics role, please send this to us by e-mail at Data Scientist Job Center

http://jobs.datschemas.org/job-detail

Job:

Job Description:

Job Title: Data Scientist

Required Skills:

- Python programming

- Machine Learning (scikit-learn, XGBoost)

- Deep Learning (TensorFlow or PyTorch)

- SQL and data manipulation

- Data visualization (

Final Explanation:


- Yes.

- No.

- If you are a data scientist, you must be a member of the Data Scientist Job Center.

- If you are not a Data Scientist Job Center member, please send us an e-mail at Data Scientist Job Center

http://jobs.datschemas.org/job-detail

Job:

Job Description:

Job Title: Data Scientist

Required Skills:

- Python programming

- Machine Learning (scikit-learn, XGBoost)

- Deep



## Step 8: Summary of All Results

In [73]:
print("SCREENING SUMMARY")
print("="*60)

for result in [result_strong, result_average, result_weak]:
    print(f"\nCandidate: {result['candidate']}")
    print(f"Score: {result['score']}")
    print(f"Explanation: {result['explanation']}")
    print("-"*60)

SCREENING SUMMARY

Candidate: Strong Candidate
Score:  (as in "This is going to be interesting")

Reply: (as in "This is going to be interesting")

Reply: (as in "I've got my first job in a data science or analytics role")

Reply: (as in "This is going to be interesting")


Example of "matching up" for all jobs:


Hello,

The following job is going to be interesting.

I'm a data scientist in an AWS role. I have a lot of experience working on big projects and I've been working very hard
Explanation: 

I am a data scientist. I am doing a lot of research into the data science fields. I want to make sure that people know my work and I am a data scientist. I have a lot of experience working on big projects and I've been working very hard.Match Analysis: Yes or noI'm currently in my mid-20s. I've been doing this job for about three years now. I am a data scientist in an AWS role. I have a lot of experience working on big projects and I've been working very hard.I am currently working on a pr

## Step 9: LangSmith Tracing

Go to https://smith.langchain.com 

## Bonus: Structured JSON Output

In [74]:
def to_json_summary(result):
    return {
        "candidate": result["candidate"],
        "fit_score": result["score"],
        "explanation": result["explanation"]
    }

summary_json = [
    to_json_summary(result_strong),
    to_json_summary(result_average),
    to_json_summary(result_weak)
]

print(json.dumps(summary_json, indent=2))

[
  {
    "candidate": "Strong Candidate",
    "fit_score": " (as in \"This is going to be interesting\")\n\nReply: (as in \"This is going to be interesting\")\n\nReply: (as in \"I've got my first job in a data science or analytics role\")\n\nReply: (as in \"This is going to be interesting\")\n\n\nExample of \"matching up\" for all jobs:\n\n\nHello,\n\nThe following job is going to be interesting.\n\nI'm a data scientist in an AWS role. I have a lot of experience working on big projects and I've been working very hard",
    "explanation": "\n\nI am a data scientist. I am doing a lot of research into the data science fields. I want to make sure that people know my work and I am a data scientist. I have a lot of experience working on big projects and I've been working very hard.Match Analysis: Yes or noI'm currently in my mid-20s. I've been doing this job for about three years now. I am a data scientist in an AWS role. I have a lot of experience working on big projects and I've been work